# Sessão 1: primeiro contato com Pandas

**Meta de hoje:** baixar o histórico de uma ação, entender o que é um `DataFrame` e calcular o primeiro retorno.

**Como usar este notebook**
- Rode cada célula em ordem com `Shift+Enter`. O resultado aparece logo abaixo.
- Células de **Exercício** têm `None  # TODO`: troque o `None` pelo seu código.
- A célula seguinte ao exercício **confere** sua resposta (✅ certo, ❌ errado, ⏳ ainda não respondido).
- Travou? Peça uma dica ao Claude. Só peça a resposta depois de tentar.
- Se algo estranho acontecer, use **Restart** (reiniciar kernel) e **Run All** (rodar tudo) para recomeçar do zero.

| Parte | O que vamos ver |
|---|---|
| 1 | Preparação |
| 2 | Baixar dados de uma ação |
| 3 | Conhecer o DataFrame |
| 4 | Exercício A: perguntas sobre os dados |
| 5 | Selecionar uma coluna (Series) |
| 6 | Exercício B: maior e menor preço |
| 7 | Retorno diário |
| 8 | Exercício C: melhor e pior dia |
| 9 | Desafio: trocar o ticker |
| 10 | Fim da sessão |

## 1. Preparação

`import` traz uma biblioteca para o seu código. O nome depois de `as` é um apelido (`pd` para Pandas é o padrão do mundo todo).

A função `conferir` no final da célula é só uma ajudante para corrigir os exercícios. Você não precisa mexer nela.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

print("Pandas versão:", pd.__version__)


def conferir(nome, resposta, esperado):
    """Compara sua resposta com a esperada e avisa o resultado."""
    if resposta is None:
        print(f"⏳ {nome}: ainda não respondido")
        return
    try:
        if isinstance(esperado, (float, np.floating)):
            certo = bool(np.isclose(float(resposta), float(esperado)))
        elif isinstance(esperado, list):
            certo = list(resposta) == esperado
        else:
            certo = bool(resposta == esperado)
    except (TypeError, ValueError):
        certo = False
    print(
        f"{'✅' if certo else '❌'} {nome}: {'certo!' if certo else 'confira de novo'}"
    )

Pandas versão: 3.0.6


## 2. Baixar os dados de uma ação

O ticker identifica a ação no Yahoo Finance:
- Bolsa brasileira (B3): termina em `.SA`, por exemplo `PETR4.SA`.
- Bolsa americana: sem sufixo, por exemplo `AAPL`.

`yf.download` devolve um **DataFrame**: uma tabela em que cada linha é um dia e cada coluna é uma informação (abertura, máxima, mínima, fechamento, volume).

O Yahoo manda as colunas em dois níveis (preço e ticker). Com `droplevel` ficamos só com o nome do preço.

In [2]:
ticker = "AAPL"
dados = yf.download(ticker, period="2y", auto_adjust=True, progress=False)
if dados.empty:
    raise ValueError(f"Não foi possível baixar os dados do ticker {ticker}")

# Simplifica os nomes das colunas (tira o nível "Ticker")
dados.columns = dados.columns.droplevel("Ticker")

dados

Price,Close,High,Low,Open,Volume
Date,,,,,
2024-10-10,227.139938,227.596129,225.285456,225.890396,28183500
2024-10-11,225.662308,227.506878,225.454043,227.397790,31759200
2024-10-14,229.381195,229.807621,226.703597,226.802758,39882100
2024-10-15,231.910049,235.519852,230.442316,231.672035,64751400
2024-10-16,229.857208,230.194384,227.933300,229.678709,34082200
...,...,...,...,...,...
2026-10-05,332.890015,336.209991,331.649994,332.820007,34400900
2026-10-06,333.630005,334.380005,330.619995,332.279999,30449000
2026-10-07,336.670013,338.670013,332.779999,336.959991,34147900


## 3. Conhecendo o DataFrame

Quatro comandos para sempre usar quando receber uma tabela nova:

| Comando | O que mostra |
|---|---|
| `head()` | as primeiras 5 linhas |
| `tail()` | as últimas 5 linhas |
| `shape` | `(linhas, colunas)` |
| `info()` | o tipo de cada coluna e se há valores faltando |
| `describe()` | estatísticas: média, mínimo, máximo... |

In [3]:
dados.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2024-10-10,227.139938,227.596129,225.285456,225.890396,28183500
2024-10-11,225.662308,227.506878,225.454043,227.397790,31759200
2024-10-14,229.381195,229.807621,226.703597,226.802758,39882100
2024-10-15,231.910049,235.519852,230.442316,231.672035,64751400
2024-10-16,229.857208,230.194384,227.933300,229.678709,34082200


In [4]:
dados.tail()

Price,Close,High,Low,Open,Volume
Date,,,,,
2026-10-05,332.890015,336.209991,331.649994,332.820007,34400900
2026-10-06,333.630005,334.380005,330.619995,332.279999,30449000
2026-10-07,336.670013,338.670013,332.779999,336.959991,34147900
2026-10-08,340.420013,341.570007,335.899994,336.820007,35332400
2026-10-09,336.640015,338.609985,330.700012,331.700012,37855000


In [5]:
print("Formato (linhas, colunas):", dados.shape)
dados.info()

Formato (linhas, colunas): (501, 5)
<class 'pandas.DataFrame'>
DatetimeIndex: 501 entries, 2024-10-10 to 2026-10-09
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   501 non-null    float64
 1   High    501 non-null    float64
 2   Low     501 non-null    float64
 3   Open    501 non-null    float64
 4   Volume  501 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 23.5 KB


In [6]:
dados.describe()

Price,Close,High,Low,Open,Volume
count,501.000000,501.000000,501.000000,501.000000,5.010000e+02
mean,254.755608,257.322529,251.923290,254.423198,5.096425e+07
std,39.085238,39.268300,38.734054,38.942419,2.258147e+07
min,171.365967,189.176417,168.175599,170.898835,1.791060e+07
25%,226.258347,228.273405,223.932796,225.612725,3.874210e+07
50%,252.169983,254.545748,249.075516,252.179977,4.567730e+07
75%,275.739410,278.771206,273.514115,276.367705,5.457510e+07
max,341.070007,345.339996,338.750000,341.079987,2.617755e+08


## 4. Exercício A: perguntas sobre os dados

Responda **escrevendo código**, não de cabeça.

- **A1.** Quantos dias (linhas) existem? *(dica: `len(...)` ou `.shape`)*
- **A2.** Qual a data do primeiro registro? E do último? *(dica: `dados.index`, com `.min()` e `.max()`)*
- **A3.** Quais são os nomes das colunas? *(dica: `dados.columns`)*

Troque cada `None` pelo seu código.

In [ ]:
n_dias = len(dados)  # TODO
primeira_data = min(dados.index)  # TODO
ultima_data = max(dados.index)  # TODO
colunas = dados.columns  # TODO

print(n_dias, primeira_data, ultima_data, colunas)


501 2024-10-10 00:00:00 2026-10-09 00:00:00 Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')


In [8]:
# Conferência (não precisa mexer)
conferir("A1 número de dias", n_dias, dados.shape[0])
conferir("A2 primeira data", primeira_data, dados.index[0])
conferir("A2 última data", ultima_data, dados.index[-1])
conferir("A3 colunas", colunas, list(dados.columns))

✅ A1 número de dias: certo!
✅ A2 primeira data: certo!
✅ A2 última data: certo!
✅ A3 colunas: certo!


## 5. Selecionando uma coluna (isso é uma Series)

Uma coluna sozinha é uma **Series**: uma lista de valores com um índice (aqui, as datas). Selecionamos pelo nome entre colchetes.

In [9]:
fechamento = dados["Close"]
print(type(fechamento))
fechamento.head()

<class 'pandas.Series'>


Date
2024-10-10    227.139938
2024-10-11    225.662308
2024-10-14    229.381195
2024-10-15    231.910049
2024-10-16    229.857208
Name: Close, dtype: float64

## 6. Exercício B: maior e menor preço

- **B1.** Qual foi o maior preço de fechamento? *(dica: `.max()`)*
- **B2.** Em que data? *(dica: `.idxmax()`)*
- **B3.** E o menor preço, e a data dele? *(dicas: `.min()` e `.idxmin()`)*

Use a Series `fechamento`.

In [10]:
maior_preco = fechamento.max()  # TODO
data_maior = fechamento.idxmax()  # TODO
menor_preco = fechamento.min()  # TODO
data_menor = fechamento.idxmin()  # TODO

print(maior_preco, data_maior, menor_preco, data_menor)

341.07000732421875 2026-09-25 00:00:00 171.365966796875 2025-04-08 00:00:00


In [11]:
# Conferência (não precisa mexer)
conferir("B1 maior preço", maior_preco, fechamento.sort_values().iloc[-1])
conferir("B2 data do maior", data_maior, fechamento.sort_values().index[-1])
conferir("B3 menor preço", menor_preco, fechamento.sort_values().iloc[0])
conferir("B3 data do menor", data_menor, fechamento.sort_values().index[0])

✅ B1 maior preço: certo!
✅ B2 data do maior: certo!
✅ B3 menor preço: certo!
✅ B3 data do menor: certo!


## 7. Retorno diário

O **retorno** diz quanto o preço variou em relação ao dia anterior. Se foi de 10 para 11, o retorno é `0.10` (10%).

`pct_change()` calcula isso para a série inteira de uma vez, sem precisar de laço (`for`). Essa forma de trabalhar, sobre a coluna toda, é a grande força do Pandas.

In [12]:
retorno = fechamento.pct_change()
retorno.head()

Date
2024-10-10         NaN
2024-10-11   -0.006505
2024-10-14    0.016480
2024-10-15    0.011025
2024-10-16   -0.008852
Name: Close, dtype: float64

**Pergunta para pensar:** por que o primeiro valor é `NaN` (vazio)? Escreva sua resposta no diário de aprendizado.

## 8. Exercício C: melhor e pior dia

- **C1.** Qual foi o melhor retorno diário? E a data? *(dicas: `.max()` e `.idxmax()`)*
- **C2.** E o pior retorno, e a data?
- **C3.** Qual o retorno médio diário? *(dica: `.mean()`)*
- **C4.** Em quantos dias o retorno foi negativo? *(dica: `retorno < 0` dá verdadeiro/falso para cada dia, e `.sum()` conta os verdadeiros)*

In [ ]:
melhor_dia = retorno.max()  # TODO
data_melhor = retorno.idxmax()  # TODO
pior_dia = retorno.min()  # TODO
data_pior = retorno.idxmin()  # TODO
media = retorno.mean()  # TODO
dias_negativos = (retorno < 0).sum()  # TODO

print(melhor_dia, data_melhor, pior_dia, data_pior, media, dias_negativos)

0.1532885035937399 2025-04-09 00:00:00 -0.09245604997985346 2025-04-03 00:00:00 0.0009497646879215424 231


In [14]:
# Conferência (não precisa mexer)
ordenado = retorno.dropna().sort_values()
conferir("C1 melhor dia", melhor_dia, ordenado.iloc[-1])
conferir("C1 data do melhor", data_melhor, ordenado.index[-1])
conferir("C2 pior dia", pior_dia, ordenado.iloc[0])
conferir("C2 data do pior", data_pior, ordenado.index[0])
conferir("C3 retorno médio", media, ordenado.sum() / len(ordenado))
conferir("C4 dias negativos", dias_negativos, len(ordenado[ordenado < 0]))

✅ C1 melhor dia: certo!
✅ C1 data do melhor: certo!
✅ C2 pior dia: certo!
✅ C2 data do pior: certo!
✅ C3 retorno médio: certo!
✅ C4 dias negativos: certo!


## 9. Desafio: troque o ticker

1. Volte à parte 2 e troque `"PETR4.SA"` por `"AAPL"` (ou `"VALE3.SA"`, `"MSFT"`...).
2. Use **Run All** (rodar tudo de novo).
3. Os resultados mudaram? E o código?

Anote no diário: o que você precisou mexer? (Resposta esperada: só o ticker.) Essa é a graça de escrever código que funciona para qualquer ação, e é exatamente o que o app final vai fazer.

## 10. Fim da sessão

Checklist (veja `guias/organizacao.md`):

- [X] O notebook roda do começo ao fim com **Restart + Run All**?
- [X] Todos os exercícios mostram ✅?
- [X] Escrevi 3 linhas no diário de aprendizado?
- [X] `git status` mostra só o que eu quero?
- [X] Commit feito (a Parte 3 do guia da sessão te ensina)?